# Fundus Vessel Segmentation — 4 model compare, best model = Proposed Model
Train: DRIVE + CHASE_DB1 (100 img) → one by one train: **U-Net++ (EffNet-B3), Attention U-Net, DeepLabV3+ (ResNet50), FR-UNet**
→ validation Dice/IoU compare → **best model automatically select** → Fundus_CIMT_2903 (5806 img) er mask generate → zip.

**Kaggle settings:** GPU ON (T4/P100), Internet ON. `QUICK_TEST = True` dile pura pipeline 2 epoch + 20 test image e dry-run hobe.

## 1. Install

In [ ]:
!pip install -q segmentation-models-pytorch albumentations timm

## 2. Config

In [ ]:
import os, re, gc, glob, json, time, random, cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

QUICK_TEST = False   # True = tutorial dry-run (2 epoch, 20 test image)

TRAIN_ROOT = "/kaggle/input/datasets/amimulhasan/fundus-mask/Fundus image segmentation"
TEST_ROOT  = "/kaggle/input/datasets/rofikbme/fundus-image/Fundus_CIMT_2903 Dataset/Fundus_CIMT_2903 Dataset"
WORK       = "/kaggle/working"
OUT_DIR    = f"{WORK}/pred_masks_best"
RESULTS_JSON = f"{WORK}/results.json"

SEED, IMG_SIZE = 42, 768
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# model-wise training config (epochs komiye time bachate paren)
CFG = {
    "UNetPlusPlus_EffB3": dict(crop=512, bs=4, epochs=60, lr=3e-4, repeat=6),
    "AttentionUNet":      dict(crop=512, bs=4, epochs=80, lr=1e-3, repeat=6),
    "DeepLabV3Plus_R50":  dict(crop=512, bs=4, epochs=60, lr=3e-4, repeat=6),
    "FR_UNet":            dict(crop=384, bs=4, epochs=60, lr=3e-4, repeat=6),
}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device:", DEVICE)

for root, dirs, files in os.walk(TRAIN_ROOT):
    depth = root[len(TRAIN_ROOT):].count(os.sep)
    if depth <= 3:
        print("  " * depth + os.path.basename(root) + f"/  ({len(files)} files)  e.g. {files[:3]}")

## 3. Image ↔ Mask pairing

In [ ]:
IMG_EXT = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".gif", ".bmp", ".ppm")
MASK_HINTS = ("mask", "manual", "1stho", "gt", "label", "vessel", "groundtruth", "annotation")

def norm_key(path):
    s = os.path.splitext(os.path.basename(path))[0].lower()
    for t in ("_manual1", "_1stho", "_mask", "_training", "_test", "_gt", "_label", "_vessel", "-mask", "_seg"):
        s = s.replace(t, "")
    return s

def is_mask(path):
    p = path.lower()
    return any(h in p for h in MASK_HINTS)

all_files = [f for f in glob.glob(TRAIN_ROOT + "/**/*", recursive=True)
             if f.lower().endswith(IMG_EXT) and "2ndho" not in f.lower()]
imgs  = [f for f in all_files if not is_mask(f)]
masks = [f for f in all_files if is_mask(f)]

mask_map = {}
for m in masks:
    k = norm_key(m)
    # DRIVE-er FOV mask (xx_training_mask) vs vessel mask (xx_manual1): vessel ta prefer koro
    if k not in mask_map or any(h in m.lower() for h in ("manual", "1stho", "vessel")):
        mask_map[k] = m

pairs = [(i, mask_map[norm_key(i)]) for i in imgs if norm_key(i) in mask_map]
print(f"images={len(imgs)}  masks={len(masks)}  paired={len(pairs)}")
for p in pairs[:5]: print(p)
assert len(pairs) > 0, "Pairing hoyni - folder structure dekhe is_mask()/norm_key() thik korte hobe"

## 4. Dataset + augmentation

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))

def preprocess(rgb):
    """Green channel + CLAHE -> 3 channel"""
    g = _clahe.apply(rgb[:, :, 1])
    return np.stack([g, g, g], axis=-1)

def read_mask(path):
    m = np.array(Image.open(path).convert("L"))
    return (m > 127).astype(np.float32)

def make_train_tf(crop):
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.RandomCrop(crop, crop),
        A.HorizontalFlip(), A.VerticalFlip(), A.RandomRotate90(),
        A.Rotate(limit=25, border_mode=cv2.BORDER_CONSTANT, p=0.5),
        A.RandomBrightnessContrast(0.15, 0.15, p=0.5),
        A.Normalize(MEAN, STD), ToTensorV2(),
    ])
val_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(MEAN, STD), ToTensorV2()])

class SegDS(Dataset):
    def __init__(self, pairs, tf, repeat=1):
        self.pairs, self.tf, self.repeat = pairs, tf, repeat
        self.cache = {}
    def __len__(self): return len(self.pairs) * self.repeat
    def __getitem__(self, i):
        i %= len(self.pairs)
        if i not in self.cache:
            ip, mp = self.pairs[i]
            img = preprocess(read_rgb(ip)); msk = read_mask(mp)
            if msk.shape != img.shape[:2]:
                msk = cv2.resize(msk, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
            self.cache[i] = (img, msk)
        img, msk = self.cache[i]
        out = self.tf(image=img, mask=msk)
        return out["image"], out["mask"].unsqueeze(0).float()

rng = random.Random(SEED)
pairs = sorted(pairs); rng.shuffle(pairs)
n_val = max(8, int(0.15 * len(pairs)))
val_pairs, train_pairs = pairs[:n_val], pairs[n_val:]
val_dl = DataLoader(SegDS(val_pairs, val_tf), batch_size=2, shuffle=False, num_workers=2)
print("train:", len(train_pairs), "val:", len(val_pairs))

## 5. Model architectures

In [ ]:
import segmentation_models_pytorch as smp

# ---------------- Attention U-Net ----------------

class ConvBlock(nn.Module):
    def __init__(self, cin, cout, p=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Dropout2d(p) if p > 0 else nn.Identity(),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class AttentionGate(nn.Module):
    """g = decoder gating signal (upsampled), x = encoder skip feature"""
    def __init__(self, f_g, f_x, f_int):
        super().__init__()
        self.Wg  = nn.Sequential(nn.Conv2d(f_g, f_int, 1, bias=False), nn.BatchNorm2d(f_int))
        self.Wx  = nn.Sequential(nn.Conv2d(f_x, f_int, 1, bias=False), nn.BatchNorm2d(f_int))
        self.psi = nn.Sequential(nn.Conv2d(f_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        a = self.psi(self.relu(self.Wg(g) + self.Wx(x)))   # (B,1,H,W) attention map
        return x * a

class UpBlock(nn.Module):
    def __init__(self, c_in, c_skip, c_out):
        super().__init__()
        self.up  = nn.Sequential(nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
                                 nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))
        self.att = AttentionGate(c_out, c_skip, c_out // 2)
        self.conv = ConvBlock(c_out + c_skip, c_out)
    def forward(self, x, skip):
        g = self.up(x)
        skip = self.att(g, skip)
        return self.conv(torch.cat([skip, g], dim=1))

class AttentionUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, f=(32, 64, 128, 256, 512)):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.e1 = ConvBlock(in_ch, f[0]); self.e2 = ConvBlock(f[0], f[1])
        self.e3 = ConvBlock(f[1], f[2]);  self.e4 = ConvBlock(f[2], f[3])
        self.bridge = ConvBlock(f[3], f[4], p=0.2)
        self.d4 = UpBlock(f[4], f[3], f[3]); self.d3 = UpBlock(f[3], f[2], f[2])
        self.d2 = UpBlock(f[2], f[1], f[1]); self.d1 = UpBlock(f[1], f[0], f[0])
        self.out = nn.Conv2d(f[0], out_ch, 1)
    def forward(self, x):
        s1 = self.e1(x); s2 = self.e2(self.pool(s1))
        s3 = self.e3(self.pool(s2)); s4 = self.e4(self.pool(s3))
        b = self.bridge(self.pool(s4))
        d = self.d4(b, s4); d = self.d3(d, s3); d = self.d2(d, s2); d = self.d1(d, s1)
        return self.out(d)

# ---------------- FR-UNet (official: github.com/lseventeen/FR-UNet, Liu et al. 2022) ----------------
class InitWeights_He(object):
    def __init__(self, neg_slope=1e-2):
        self.neg_slope = neg_slope
    def __call__(self, module):
        if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
            module.weight = nn.init.kaiming_normal_(module.weight, a=self.neg_slope)
            if module.bias is not None:
                module.bias = nn.init.constant_(module.bias, 0)

class conv(nn.Module):
    def __init__(self, in_c, out_c, dp=0):
        super(conv, self).__init__()
        self.in_c = in_c
        self.out_c = out_c
        self.conv = nn.Sequential(
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.Dropout2d(dp),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.Dropout2d(dp),
            nn.LeakyReLU(0.1, inplace=True))
        

    def forward(self, x):
        return self.conv(x)


class feature_fuse(nn.Module):
    def __init__(self, in_c, out_c):
        super(feature_fuse, self).__init__()
        self.conv11 = nn.Conv2d(
            in_c, out_c, kernel_size=1, padding=0, bias=False)
        self.conv33 = nn.Conv2d(
            in_c, out_c, kernel_size=3, padding=1, bias=False)
        self.conv33_di = nn.Conv2d(
            in_c, out_c, kernel_size=3, padding=2, bias=False, dilation=2)
        self.norm = nn.BatchNorm2d(out_c)

    def forward(self, x):
        x1 = self.conv11(x)
        x2 = self.conv33(x)
        x3 = self.conv33_di(x)
        out = self.norm(x1+x2+x3)
        return out


class up(nn.Module):
    def __init__(self, in_c, out_c, dp=0):
        super(up, self).__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c, kernel_size=2,
                               padding=0, stride=2, bias=False),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.1, inplace=False))

    def forward(self, x):
        x = self.up(x)
        return x


class down(nn.Module):
    def __init__(self, in_c, out_c, dp=0):
        super(down, self).__init__()
        self.down = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=2,
                      padding=0, stride=2, bias=False),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.1, inplace=True))

    def forward(self, x):
        x = self.down(x)
        return x


class block(nn.Module):
    def __init__(self, in_c, out_c,  dp=0, is_up=False, is_down=False, fuse=False):
        super(block, self).__init__()
        self.in_c = in_c
        self.out_c = out_c
        if fuse == True:
            self.fuse = feature_fuse(in_c, out_c)
        else:
            self.fuse = nn.Conv2d(in_c, out_c, kernel_size=1, stride=1)

        self.is_up = is_up
        self.is_down = is_down
        self.conv = conv(out_c, out_c, dp=dp)
        if self.is_up == True:
            self.up = up(out_c, out_c//2)
        if self.is_down == True:
            self.down = down(out_c, out_c*2)

    def forward(self,  x):
        if self.in_c != self.out_c:
            x = self.fuse(x)
        x = self.conv(x)
        if self.is_up == False and self.is_down == False:
            return x
        elif self.is_up == True and self.is_down == False:
            x_up = self.up(x)
            return x, x_up
        elif self.is_up == False and self.is_down == True:
            x_down = self.down(x)
            return x, x_down
        else:
            x_up = self.up(x)
            x_down = self.down(x)
            return x, x_up, x_down


class FR_UNet(nn.Module):
    def __init__(self,  num_classes=1, num_channels=1, feature_scale=2,  dropout=0.2, fuse=True, out_ave=True):
        super(FR_UNet, self).__init__()
        self.out_ave = out_ave
        filters = [64, 128, 256, 512, 1024]
        filters = [int(x / feature_scale) for x in filters]
        self.block1_3 = block(
            num_channels, filters[0],  dp=dropout, is_up=False, is_down=True, fuse=fuse)
        self.block1_2 = block(
            filters[0], filters[0],  dp=dropout, is_up=False, is_down=True, fuse=fuse)
        self.block1_1 = block(
            filters[0]*2, filters[0],  dp=dropout, is_up=False, is_down=True, fuse=fuse)
        self.block10 = block(
            filters[0]*2, filters[0],  dp=dropout, is_up=False, is_down=True, fuse=fuse)
        self.block11 = block(
            filters[0]*2, filters[0],  dp=dropout, is_up=False, is_down=True, fuse=fuse)
        self.block12 = block(
            filters[0]*2, filters[0],  dp=dropout, is_up=False, is_down=False, fuse=fuse)
        self.block13 = block(
            filters[0]*2, filters[0],  dp=dropout, is_up=False, is_down=False, fuse=fuse)
        self.block2_2 = block(
            filters[1], filters[1],  dp=dropout, is_up=True, is_down=True, fuse=fuse)
        self.block2_1 = block(
            filters[1]*2, filters[1],  dp=dropout, is_up=True, is_down=True, fuse=fuse)
        self.block20 = block(
            filters[1]*3, filters[1],  dp=dropout, is_up=True, is_down=True, fuse=fuse)
        self.block21 = block(
            filters[1]*3, filters[1],  dp=dropout, is_up=True, is_down=False, fuse=fuse)
        self.block22 = block(
            filters[1]*3, filters[1],  dp=dropout, is_up=True, is_down=False, fuse=fuse)
        self.block3_1 = block(
            filters[2], filters[2],  dp=dropout, is_up=True, is_down=True, fuse=fuse)
        self.block30 = block(
            filters[2]*2, filters[2],  dp=dropout, is_up=True, is_down=False, fuse=fuse)
        self.block31 = block(
            filters[2]*3, filters[2],  dp=dropout, is_up=True, is_down=False, fuse=fuse)
        self.block40 = block(filters[3], filters[3],
                             dp=dropout, is_up=True, is_down=False, fuse=fuse)
        self.final1 = nn.Conv2d(
            filters[0], num_classes, kernel_size=1, padding=0, bias=True)
        self.final2 = nn.Conv2d(
            filters[0], num_classes, kernel_size=1, padding=0, bias=True)
        self.final3 = nn.Conv2d(
            filters[0], num_classes, kernel_size=1, padding=0, bias=True)
        self.final4 = nn.Conv2d(
            filters[0], num_classes, kernel_size=1, padding=0, bias=True)
        self.final5 = nn.Conv2d(
            filters[0], num_classes, kernel_size=1, padding=0, bias=True)
        self.fuse = nn.Conv2d(
            5, num_classes, kernel_size=1, padding=0, bias=True)
        self.apply(InitWeights_He)

    def forward(self, x):
        x1_3, x_down1_3 = self.block1_3(x)
        x1_2, x_down1_2 = self.block1_2(x1_3)
        x2_2, x_up2_2, x_down2_2 = self.block2_2(x_down1_3)
        x1_1, x_down1_1 = self.block1_1(torch.cat([x1_2, x_up2_2], dim=1))
        x2_1, x_up2_1, x_down2_1 = self.block2_1(
            torch.cat([x_down1_2, x2_2], dim=1))
        x3_1, x_up3_1, x_down3_1 = self.block3_1(x_down2_2)
        x10, x_down10 = self.block10(torch.cat([x1_1, x_up2_1], dim=1))
        x20, x_up20, x_down20 = self.block20(
            torch.cat([x_down1_1, x2_1, x_up3_1], dim=1))
        x30, x_up30 = self.block30(torch.cat([x_down2_1, x3_1], dim=1))
        _, x_up40 = self.block40(x_down3_1)
        x11, x_down11 = self.block11(torch.cat([x10, x_up20], dim=1))
        x21, x_up21 = self.block21(torch.cat([x_down10, x20, x_up30], dim=1))
        _, x_up31 = self.block31(torch.cat([x_down20, x30, x_up40], dim=1))
        x12 = self.block12(torch.cat([x11, x_up21], dim=1))
        _, x_up22 = self.block22(torch.cat([x_down11, x21, x_up31], dim=1))
        x13 = self.block13(torch.cat([x12, x_up22], dim=1))
        if self.out_ave == True:
            output = (self.final1(x1_1)+self.final2(x10) +
                      self.final3(x11)+self.final4(x12)+self.final5(x13))/5
        else:
            output = self.final5(x13)

        return output

# ---------------- model factory ----------------
def build_model(name):
    if name == "UNetPlusPlus_EffB3":
        try:
            return smp.UnetPlusPlus(encoder_name="tu-efficientnet_b3", encoder_weights="imagenet", in_channels=3, classes=1)
        except Exception as e:
            print("tu-efficientnet_b3 fail, fallback efficientnet-b3:", repr(e)[:150])
            return smp.UnetPlusPlus(encoder_name="efficientnet-b3", encoder_weights="imagenet", in_channels=3, classes=1)
    if name == "AttentionUNet":
        return AttentionUNet()
    if name == "DeepLabV3Plus_R50":
        return smp.DeepLabV3Plus(encoder_name="resnet50", encoder_weights="imagenet", encoder_output_stride=16, in_channels=3, classes=1)
    if name == "FR_UNet":
        return FR_UNet(num_classes=1, num_channels=3, feature_scale=2, dropout=0.2, fuse=True, out_ave=True)
    raise ValueError(name)

for n in CFG:
    m = build_model(n)
    print(f"{n:22s} params = {sum(p.numel() for p in m.parameters())/1e6:.2f}M")
    del m

## 6. Loss, metrics, Dual-Threshold post-processing, training function

In [ ]:
class DiceLoss(nn.Module):
    def forward(self, logits, target, eps=1e-6):
        p = torch.sigmoid(logits)
        inter = (p * target).sum((1, 2, 3))
        dice = (2 * inter + eps) / (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + eps)
        return 1 - dice.mean()

dice_loss, bce_loss = DiceLoss(), nn.BCEWithLogitsLoss()
def criterion(p, t): return 0.5 * bce_loss(p, t) + 0.5 * dice_loss(p, t)

@torch.no_grad()
def dice_iou(logits, target, thr=0.5, eps=1e-6):
    p = (torch.sigmoid(logits) > thr).float()
    inter = (p * target).sum((1, 2, 3))
    dice = (2 * inter + eps) / (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + eps)
    iou  = (inter + eps) / (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) - inter + eps)
    return dice.mean().item(), iou.mean().item()

# ---- Dual-Threshold Iteration (high threshold seed + low threshold connected growth) ----
def dual_threshold(prob, tl, th):
    high = prob >= th
    low = (prob >= tl).astype(np.uint8)
    n, lab = cv2.connectedComponents(low, connectivity=8)
    keep = np.unique(lab[high]); keep = keep[keep > 0]
    return np.isin(lab, keep).astype(np.uint8)

@torch.no_grad()
def collect_val_probs(model):
    model.eval(); probs, gts = [], []
    for x, y in val_dl:
        p = torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()[:, 0]
        probs += list(p); gts += list(y.numpy()[:, 0])
    return probs, gts

def score(probs, gts, tl, th):
    ds, ios = [], []
    for p, g in zip(probs, gts):
        m = dual_threshold(p, tl, th)
        inter = (m * g).sum()
        ds.append((2 * inter + 1e-6) / (m.sum() + g.sum() + 1e-6))
        ios.append((inter + 1e-6) / (m.sum() + g.sum() - inter + 1e-6))
    return float(np.mean(ds)), float(np.mean(ios))

def tune_thresholds(probs, gts):
    base = score(probs, gts, 0.5, 0.5)
    best = (base[0], base[1], 0.5, 0.5)
    for tl in (0.3, 0.35, 0.4, 0.45, 0.5):
        for th in (0.5, 0.6, 0.7, 0.8, 0.9):
            if tl <= th:
                d, i = score(probs, gts, tl, th)
                if d > best[0]: best = (d, i, tl, th)
    return base, best

def train_model(name):
    cfg = CFG[name]; n_ep = 2 if QUICK_TEST else cfg["epochs"]
    ckpt = f"{WORK}/ckpt_{name}.pth"
    model = build_model(name).to(DEVICE)
    train_dl = DataLoader(SegDS(train_pairs, make_train_tf(cfg["crop"]), cfg["repeat"]),
                          batch_size=cfg["bs"], shuffle=True, num_workers=2, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
    best, hist, t0 = 0.0, [], time.time()
    for ep in range(1, n_ep + 1):
        model.train(); tl_sum = 0
        for x, y in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl_sum += loss.item()
        sched.step()
        model.eval(); ds, ios = [], []
        with torch.no_grad():
            for x, y in val_dl:
                d, i = dice_iou(model(x.to(DEVICE)), y.to(DEVICE)); ds.append(d); ios.append(i)
        vd, vi = float(np.mean(ds)), float(np.mean(ios)); hist.append(vd)
        flag = ""
        if vd > best:
            best = vd; torch.save(model.state_dict(), ckpt); flag = " *saved"
        print(f"[{name}] ep {ep:03d}/{n_ep} | loss {tl_sum/len(train_dl):.4f} | val Dice {vd:.4f} IoU {vi:.4f}{flag}")
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); model.eval()
    return model, dict(history=hist, train_min=round((time.time() - t0) / 60, 1),
                       params_M=round(sum(p.numel() for p in model.parameters()) / 1e6, 2))

## 7. Train all models one by one (+ threshold tuning on validation)

In [ ]:
results = json.load(open(RESULTS_JSON)) if (os.path.exists(RESULTS_JSON) and not QUICK_TEST) else {}

for name in CFG:
    ckpt = f"{WORK}/ckpt_{name}.pth"
    if name in results and os.path.exists(ckpt):
        print(f"== {name}: already trained, skip =="); continue
    print(f"\n================ TRAINING: {name} ================")
    model, info = train_model(name)
    probs, gts = collect_val_probs(model)
    base, best = tune_thresholds(probs, gts)
    results[name] = dict(
        dice_thr05=round(base[0], 4), iou_thr05=round(base[1], 4),
        dice_tuned=round(best[0], 4), iou_tuned=round(best[1], 4),
        TL=best[2], TH=best[3], **info)
    print(f">>> {name}: Dice@0.5={base[0]:.4f} | tuned Dice={best[0]:.4f} (TL={best[2]}, TH={best[3]}) | {info['train_min']} min")
    json.dump(results, open(RESULTS_JSON, "w"), indent=1)
    del model; gc.collect(); torch.cuda.empty_cache()

## 8. Comparison table + select Proposed Model

In [ ]:
import matplotlib.pyplot as plt

df = pd.DataFrame({k: {c: v[c] for c in ["params_M", "train_min", "dice_thr05", "iou_thr05", "dice_tuned", "iou_tuned", "TL", "TH"]}
                   for k, v in results.items()}).T.sort_values("dice_tuned", ascending=False)
display(df)
df.to_csv(f"{WORK}/model_comparison.csv")

BEST_NAME = df.index[0]
BEST_TL, BEST_TH = float(df.loc[BEST_NAME, "TL"]), float(df.loc[BEST_NAME, "TH"])
open(f"{WORK}/proposed_model.txt", "w").write(f"{BEST_NAME}\nval_dice={df.loc[BEST_NAME,'dice_tuned']}\nTL={BEST_TL}\nTH={BEST_TH}\n")
print(f"\n PROPOSED MODEL = {BEST_NAME}  (val Dice {df.loc[BEST_NAME,'dice_tuned']}, TL={BEST_TL}, TH={BEST_TH})")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for k, v in results.items():
    ax[0].plot(range(1, len(v["history"]) + 1), v["history"], label=k)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("val Dice"); ax[0].legend(); ax[0].set_title("Validation Dice curves")
ax[1].bar(df.index, df["dice_tuned"].astype(float)); ax[1].set_ylim(max(0, df["dice_tuned"].astype(float).min() - 0.05), 1)
ax[1].set_title("Final val Dice (tuned threshold)"); plt.setp(ax[1].get_xticklabels(), rotation=20, ha="right")
plt.tight_layout(); plt.show()

## 9. Proposed (best) model diye 5806 image er mask generate

In [ ]:
model = build_model(BEST_NAME).to(DEVICE)
model.load_state_dict(torch.load(f"{WORK}/ckpt_{BEST_NAME}.pth", map_location=DEVICE)); model.eval()
os.makedirs(OUT_DIR, exist_ok=True)

test_files = sorted(f for f in glob.glob(TEST_ROOT + "/**/*", recursive=True) if f.lower().endswith(IMG_EXT))
if QUICK_TEST: test_files = test_files[:20]
print("test images:", len(test_files))
norm = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(MEAN, STD), ToTensorV2()])

def fov_mask(rgb, thr=15):
    g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    m = (g > thr).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    return cv2.erode(m, np.ones((15, 15), np.uint8))

@torch.no_grad()
def predict_prob(rgb):
    x = norm(image=preprocess(rgb))["image"].unsqueeze(0).to(DEVICE)
    p = torch.sigmoid(model(x))
    p += torch.sigmoid(model(x.flip(-1))).flip(-1)
    p += torch.sigmoid(model(x.flip(-2))).flip(-2)
    p += torch.sigmoid(model(x.flip(-1, -2))).flip(-1, -2)
    return (p / 4)[0, 0].cpu().numpy()

for f in tqdm(test_files):
    rgb = read_rgb(f); h, w = rgb.shape[:2]
    prob = cv2.resize(predict_prob(rgb), (w, h), interpolation=cv2.INTER_LINEAR)
    m = dual_threshold(prob, BEST_TL, BEST_TH)
    mask = ((m > 0) & (fov_mask(rgb) > 0)).astype(np.uint8) * 255
    name = os.path.splitext(os.path.basename(f))[0]
    cv2.imwrite(f"{OUT_DIR}/{name}_mask.png", mask)
print("Done ->", OUT_DIR, "| masks:", len(os.listdir(OUT_DIR)))

## 10. Visual check

In [ ]:
for f in random.sample(test_files, min(3, len(test_files))):
    name = os.path.splitext(os.path.basename(f))[0]
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(read_rgb(f)); ax[0].axis("off")
    ax[1].imshow(Image.open(f"{OUT_DIR}/{name}_mask.png"), cmap="gray"); ax[1].axis("off")
    plt.show()

## 11. Masks zip

In [ ]:
import shutil
ZIP_BASE = f"{WORK}/fundus_pred_masks_BEST"
shutil.make_archive(ZIP_BASE, "zip", OUT_DIR)
print("ZIP ready:", ZIP_BASE + ".zip", round(os.path.getsize(ZIP_BASE + ".zip") / 1e6, 1), "MB")
shutil.rmtree(OUT_DIR)   # output panel clean rakhte
print(sorted(os.listdir(WORK)))